## Day 1: Agents vs. Chatbots, and the ReAct Loop

A chatbot maps one message to one reply. An **agent** repeats an
observe -> reason -> act cycle until it reaches a goal or a stop
condition. Below is a from-scratch ReAct-style loop for a different
scenario than the daily notes: a small agent that answers currency
questions using a `convert_currency` tool instead of doing the math
itself. It also shows the few-shot priming and the max-step guard.

In [ ]:
FEW_SHOT_PRIMER = '''
Question: How many euros is 100 US dollars?
Thought: I need the current USD->EUR rate before I can answer.
Action: convert_currency(100, "USD", "EUR")
Observation: 92.10
Thought: I now have enough to answer.
Final Answer: About 92.10 EUR.
'''

def convert_currency(amount, src, dst):
    # Stand-in for a real FX-rate lookup.
    fake_rates = {("USD", "EUR"): 0.921, ("USD", "JPY"): 149.2}
    rate = fake_rates.get((src, dst), 1.0)
    return round(amount * rate, 2)

TOOLS = {"convert_currency": convert_currency}

def parse_action(line):
    # Expects: Action: convert_currency(100, "USD", "EUR")
    name = line.split("Action:")[1].split("(")[0].strip()
    raw_args = line.split("(", 1)[1].rsplit(")", 1)[0]
    parts = [p.strip().strip('"') for p in raw_args.split(",")]
    amount, src, dst = float(parts[0]), parts[1], parts[2]
    return name, (amount, src, dst)

def run_currency_agent(fake_model_generate, question, max_steps=5):
    transcript = FEW_SHOT_PRIMER + f"\nQuestion: {question}\n"
    for step in range(max_steps):  # hard safety cap on loop iterations
        chunk = fake_model_generate(transcript)
        transcript += chunk
        if "Final Answer:" in chunk:
            return chunk.split("Final Answer:")[-1].strip()
        action_line = next(l for l in chunk.splitlines() if l.startswith("Action:"))
        name, (amount, src, dst) = parse_action(action_line)
        result = TOOLS[name](amount, src, dst)
        transcript += f"Observation: {result}\n"
    return "Stopped: exceeded max_steps without a Final Answer."

In [ ]:
# Two mock model behaviors to illustrate the before/after contrast from
# the written notes, using the currency scenario instead of a math one.

def plain_chatbot_mock(prompt):
    # A chatbot with no tool access just guesses fluently -- and can be wrong.
    return "Final Answer: Roughly 100 EUR (same as the dollar amount)."

def react_mock_generate(transcript):
    # A stand-in for a real local model call, e.g.:
    #   tokenizer(transcript, return_tensors='pt')
    #   model.generate(**inputs, max_new_tokens=64, stop_strings=['Observation:'])
    if "Observation:" not in transcript:
        return 'Thought: I need the rate.\nAction: convert_currency(100, "USD", "EUR")\n'
    return "Thought: Done.\nFinal Answer: 92.10 EUR.\n"

print(plain_chatbot_mock("How many euros is 100 US dollars?"))
print(run_currency_agent(react_mock_generate, "How many euros is 100 US dollars?"))

## Day 2: Agent Framework Landscape and a Tiny Framework

The wider agent-tooling ecosystem roughly splits into autonomous
runtimes, sandboxing/security wrappers, and framework-agnostic
orchestration layers on top of several engines. Rather than reusing
the `MiniAgentFramework` example from the notes, this section builds a
small `ToolRegistry` (tool bookkeeping + audit log) around a document-
translation scenario, and a separate LCEL-style pipe chain for it.

In [ ]:
import time

class ToolRegistry:
    """A minimal stand-in for what larger agent frameworks manage:
    a table of callable tools plus a running audit log."""

    def __init__(self):
        self._tools = {}
        self.audit_log = []

    def register_tool(self, name, fn, allowed=True):
        self._tools[name] = {"fn": fn, "allowed": allowed}

    def run_tool(self, name, **kwargs):
        record = {"tool": name, "kwargs": kwargs, "ts": time.time()}
        entry = self._tools.get(name)
        if entry is None or not entry["allowed"]:
            record["status"] = "denied"
            self.audit_log.append(record)
            raise PermissionError(f"tool '{name}' is not available")
        record["status"] = "ok"
        record["result"] = entry["fn"](**kwargs)
        self.audit_log.append(record)
        return record["result"]

def detect_language(text):
    return "fr" if text.lower().startswith("bonjour") else "en"

registry = ToolRegistry()
registry.register_tool("detect_language", detect_language)
print(registry.run_tool("detect_language", text="Bonjour tout le monde"))
print(registry.audit_log)

In [ ]:
# A schematic LCEL-style pipe chain (prompt | model | parser) for a
# translation task -- illustrative, not a real LangChain install.
# A real integration would subclass a real base class, e.g. LangChain's
# `LLM` class, and implement its `_call` method.

class RunnableStep:
    def __or__(self, other):
        return PipedStep(self, other)

class PipedStep(RunnableStep):
    def __init__(self, first, second):
        self.first, self.second = first, second

    def invoke(self, x):
        return self.second.invoke(self.first.invoke(x))

class PromptStep(RunnableStep):
    def __init__(self, template):
        self.template = template

    def invoke(self, variables):
        return self.template.format(**variables)

class LocalLLMStep(RunnableStep):
    def __init__(self, generate_fn):
        self.generate_fn = generate_fn

    def invoke(self, prompt_text):
        return self.generate_fn(prompt_text)

class StripParserStep(RunnableStep):
    def invoke(self, raw_text):
        return raw_text.strip()

translate_prompt = PromptStep("Translate to Spanish: {text}")
mock_llm = LocalLLMStep(lambda p: "  Hola a todos  ")
translate_chain = translate_prompt | mock_llm | StripParserStep()
print(translate_chain.invoke({"text": "Hello everyone"}))

## Day 3: Local LLMs and Quantization

Cloud models trade privacy and cost-control for scale; local models
trade scale for privacy, offline use, and no per-token bill. This
section uses a support-ticket triage scenario (different from the
notes' plain arithmetic example) to show subprocess-isolated dtype
measurement, a CPU-only quantization fallback, and a tool-call
convention parsed out of raw model text.

In [ ]:
import subprocess
import sys

def measure_load_time(model_id, dtype_name):
    """Runs the model load in its own fresh subprocess so one dtype's
    allocator/cache state can't bleed into the next measurement."""
    probe_script = (
        "import time, torch\n"
        "from transformers import AutoModelForCausalLM\n"
        "t0 = time.time()\n"
        f"model = AutoModelForCausalLM.from_pretrained("
        f"'{model_id}', torch_dtype=torch.{dtype_name})\n"
        "print(f'{time.time() - t0:.2f}s')\n"
    )
    completed = subprocess.run(
        [sys.executable, "-c", probe_script],
        capture_output=True,
        text=True,
    )
    return completed.stdout.strip() or completed.stderr.strip()

# Requires `transformers` + `torch` and a downloaded checkpoint to actually
# run; shown here for the correct call shape, not for live execution.
for dtype in ["float32", "float16", "bfloat16"]:
    result = measure_load_time("support-triage-tinylm", dtype)
    print(dtype, "->", result)

In [ ]:
import re

# A CPU-only fallback case study: an 8-bit quantization step that
# requires a CUDA backend and should fail gracefully, not crash.
def build_quantized_layer(in_features, out_features):
    try:
        import bitsandbytes as bnb
        return bnb.nn.Linear8bitLt(
            in_features, out_features, has_fp16_weights=False
        )
    except Exception as err:
        print(f"quantized layer unavailable ({err}); using full precision instead")
        import torch
        return torch.nn.Linear(in_features, out_features)

# A tool-call convention: the model is instructed to emit lines like
#   TOOL_CALL: lookup_order_status(order_id=4471)
# which Python then extracts with a regex and dispatches.
TOOL_CALL_PATTERN = re.compile(r"TOOL_CALL:\s*(\w+)\((.*)\)")

def lookup_order_status(order_id):
    return {"order_id": order_id, "status": "shipped"}

SUPPORT_TOOLS = {"lookup_order_status": lookup_order_status}

def dispatch_from_model_text(model_text):
    match = TOOL_CALL_PATTERN.search(model_text)
    if not match:
        return None
    tool_name, raw_args = match.group(1), match.group(2)
    order_id = int(raw_args.split("=")[1])
    return SUPPORT_TOOLS[tool_name](order_id=order_id)

sample_model_output = "TOOL_CALL: lookup_order_status(order_id=4471)"
print(dispatch_from_model_text(sample_model_output))

## Day 4: Agent Safety Mechanisms

Four guardrails -- tool allowlisting (default-deny), a step limit, a
human-approval gate, and a cost cap -- combined into one guarded
runner, checked cheapest-first. This section uses a social-media
posting agent (different tools than the notes' example) plus an
audit-trail anomaly check.

In [ ]:
import time

class SafeAgentGate:
    """Combines four guardrails, checked cheapest/fastest first:
    step limit and cost cap are simple counter comparisons, the
    allowlist is a slightly pricier dict lookup, and human approval
    runs last because it can block on a real person's response."""

    def __init__(self, allowed_tools, max_steps, budget_usd, approve_fn):
        self.allowed_tools = set(allowed_tools)
        self.max_steps = max_steps
        self.budget_usd = budget_usd
        self.spent_usd = 0.0
        self.steps_taken = 0
        self.approve_fn = approve_fn
        self.audit_rows = []

    def attempt(self, tool_name, args, est_cost, needs_approval=False):
        row = {
            "timestamp": time.time(),
            "tool": tool_name,
            "args": str(args),
            "cost": est_cost,
            "decision": None,
        }
        if self.steps_taken >= self.max_steps:
            row["decision"] = "denied_step_limit"
        elif self.spent_usd + est_cost > self.budget_usd:
            row["decision"] = "denied_cost_cap"
        elif tool_name not in self.allowed_tools:
            row["decision"] = "denied_not_allowlisted"
        elif needs_approval and not self.approve_fn(tool_name, args):
            row["decision"] = "denied_no_approval"
        else:
            row["decision"] = "allowed"
            self.steps_taken += 1
            self.spent_usd += est_cost
        self.audit_rows.append(row)
        return row["decision"] == "allowed"

def post_to_social_media(caption):
    return f"posted: {caption}"

def refund_customer(order_id, amount):
    return f"refunded {amount} for order {order_id}"

def always_deny(tool_name, args):
    return False  # stand-in for a real human-approval prompt

gate = SafeAgentGate(
    allowed_tools={"post_to_social_media"},
    max_steps=10,
    budget_usd=5.00,
    approve_fn=always_deny,
)
print(gate.attempt("post_to_social_media", {"caption": "New product!"}, est_cost=0.02))
print(gate.attempt("refund_customer", {"order_id": 91, "amount": 40}, est_cost=0.01, needs_approval=True))

In [ ]:
import pandas as pd

audit_df = pd.DataFrame(gate.audit_rows)

# Flag (tool, args) combinations attempted an unusually high number of
# times -- a simple heuristic for a stuck or misbehaving agent, no ML
# model required.
repeat_counts = (
    audit_df.groupby(["tool", "args"])
    .size()
    .reset_index(name="attempts")
)
flagged = repeat_counts[repeat_counts["attempts"] >= 5]
print(audit_df)
print(flagged)

# Model-routing pattern (qualitative, no invented figures): route routine
# steps to a cheap local model and escalate to a larger cloud model only
# for one best-effort attempt once the local agent is stuck or out of
# steps -- most requests stay on the cheaper path, and the hard cases
# still get a shot at a more capable model.
def route_request(local_agent_run, cloud_agent_run, question):
    local_result = local_agent_run(question)
    if local_result is None:
        return cloud_agent_run(question)  # escalate only when stuck
    return local_result